In [5]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_fscore_support, f1_score, average_precision_score

# Đọc dữ liệu thống kê gốc
data_path = '/content/drive/MyDrive/Colab Notebooks/data/honduras_features.csv'
df = pd.read_csv(data_path)

# Tách features đầy đủ và features đã bỏ tweet_count
features_full = ['tweet_count', 'avg_text_length', 'url_ratio', 'hashtag_ratio', 'mention_ratio', 'retweet_ratio']
features_no_tc = ['avg_text_length', 'url_ratio', 'hashtag_ratio', 'mention_ratio', 'retweet_ratio']

train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'validation'].copy()
test_df = df[df['split'] == 'test'].copy()

y_train, y_val, y_test = train_df['label'], val_df['label'], test_df['label']

def train_and_eval_lr(features_list, model_name):
    # Chuẩn hóa
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[features_list])
    X_test = scaler.transform(test_df[features_list])

    # Huấn luyện LR với class_weight='balanced'
    lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
    lr.fit(X_train, y_train)

    # Đánh giá trên tập Test
    y_pred = lr.predict(X_test)
    y_prob = lr.predict_proba(X_test)[:, 0] # Xác suất nhãn 0 (Bad)

    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1], zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    ap = average_precision_score((y_test == 0).astype(int), y_prob)

    return {'Method': model_name, 'Bad-Precision': p[0], 'Bad-Recall': r[0], 'Bad-F1': f1[0], 'Macro-F1': macro_f1, 'AP': ap}

# Tập hợp kết quả Baseline & Ablation
results = []
results.append(train_and_eval_lr(features_full, 'LR (Full Features)'))
results.append(train_and_eval_lr(features_no_tc, 'LR (No tweet_count)'))

results_df = pd.DataFrame(results)
display(results_df)

,Method,Bad-Precision,Bad-Recall,Bad-F1,Macro-F1,AP
0,LR (Full Features),0.089131,0.965147,0.163191,0.559996,0.471164
1,LR (No tweet_count),0.086341,0.981233,0.158716,0.556605,0.368820


In [6]:
import torch
import torch.nn as nn

# Cấu trúc Encoder (Phải khớp hoàn toàn với E1)
class BehavioralEncoder(nn.Module):
    def __init__(self, input_dim=10000, hidden_dim=256, embedding_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, embedding_dim),
            nn.ReLU(),
        )
    def forward(self, x):
        return self.encoder(x)

# Mô hình phân loại E2 ghép thêm Classification Head
class HondurasBotClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        # Head phân loại Bad (0) / Good (1)
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1) # Binary classification
        )

    def forward(self, x):
        emb = self.encoder(x)
        logits = self.head(emb)
        return logits

# 1. KHỞI TẠO MÔ HÌNH SCRATCH (Trọng số ngẫu nhiên)
encoder_scratch = BehavioralEncoder()
model_scratch = HondurasBotClassifier(encoder_scratch)

# 2. KHỞI TẠO MÔ HÌNH TRANSFER
CHECKPOINT_PATH = '/content/drive/MyDrive/Colab Notebooks/data/E1/e1_behavioral_encoder.pt'
checkpoint = torch.load(CHECKPOINT_PATH, map_location=torch.device('cpu'))

encoder_transfer = BehavioralEncoder(
    input_dim=int(checkpoint['input_dim']),
    hidden_dim=int(checkpoint['hidden_dim']),
    embedding_dim=int(checkpoint['embedding_dim'])
)
# Nạp tạ từ Fox8
encoder_transfer.encoder.load_state_dict(checkpoint['encoder_state_dict'])
model_transfer = HondurasBotClassifier(encoder_transfer)

print("Đã khởi tạo kiến trúc cho Train from Scratch và Transfer Learning.")

Đã khởi tạo kiến trúc cho Train from Scratch và Transfer Learning.


In [7]:
import json
from collections import defaultdict
import pandas as pd

# 1. Đường dẫn tới 2 file văn bản gốc của Honduras
bad_jsonl = '/content/drive/MyDrive/Colab Notebooks/data/honduras-bad-anonymized'
good_jsonl = '/content/drive/MyDrive/Colab Notebooks/data/honduras-good-anonymized'

# 2. Đọc và gom tối đa 30 tweets per account_id (chuẩn E1)
account_texts = defaultdict(list)
for path in [bad_jsonl, good_jsonl]:
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            # Sửa lại khóa cho phù hợp với dữ liệu thực tế: 'userid' và 'tweet_text'
            acc_id = str(rec.get('userid') or rec.get('account_id') or rec.get('user_id') or rec.get('id') or '').strip()
            text = rec.get('tweet_text') or rec.get('text')
            if acc_id and isinstance(text, str) and text.strip():
                if len(account_texts[acc_id]) < 30:
                    account_texts[acc_id].append(text.strip())

# Ghép văn bản cho từng tài khoản (giới hạn 5000 ký tự)
text_data = [{'account_id': acc_id, 'text': " ".join(texts)[:5000]} for acc_id, texts in account_texts.items()]
df_text = pd.DataFrame(text_data)

# Khởi tạo cột account_id và text mặc định nếu df_text bị rỗng để tránh lỗi merge
if df_text.empty:
    df_text = pd.DataFrame(columns=['account_id', 'text'])
else:
    # Đảm bảo account_id cùng kiểu dữ liệu với df gốc (object/string hoặc int64)
    df_text['account_id'] = df_text['account_id'].astype(str)

# 3. Đọc dữ liệu features gốc
data_path = '/content/drive/MyDrive/Colab Notebooks/data/honduras_features.csv'
df = pd.read_csv(data_path)

# Đảm bảo account_id là kiểu string để merge chuẩn xác
df['account_id'] = df['account_id'].astype(str)

# Đảm bảo account_id là một cột (chuyển về cột nếu lỡ nằm ở Index)
if 'account_id' not in df.columns:
    df = df.reset_index()

# Merge cột 'text' vào df tổng trước khi chia tập
df = pd.merge(df, df_text, on='account_id', how='left')
df['text'] = df['text'].fillna('')

# 4. Chia tập train / validation / test đã có sẵn cột 'text'
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'validation'].copy()
test_df = df[df['split'] == 'test'].copy()

y_train = train_df['label']
y_val = val_df['label']
y_test = test_df['label']

print(f"Đã bổ sung văn bản cho {len(df_text)} tài khoản thành công!")
print(f"Số mẫu Train: {len(train_df)} | Test: {len(test_df)}")

Đã bổ sung văn bản cho 224685 tài khoản thành công!
Số mẫu Train: 157279 | Test: 44938


In [8]:
import gc
import os
import torch
import joblib
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Cập nhật các tập dữ liệu con từ df tổng
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'validation'].copy()
test_df = df[df['split'] == 'test'].copy()

y_train = train_df['label']
y_val = val_df['label']
y_test = test_df['label']

# 3. Nạp TF-IDF Vectorizer từ E1
vectorizer_path = '/content/drive/MyDrive/Colab Notebooks/data/E1/tfidf_vectorizer.joblib'
vectorizer = joblib.load(vectorizer_path)

# 4. Biến đổi văn bản sang ma trận thưa
X_train_sparse = vectorizer.transform(train_df['text'].fillna(''))
X_test_sparse = vectorizer.transform(test_df['text'].fillna(''))

# 5. Hàm DataLoader chuyển đổi dense theo từng batch nhỏ (On-the-fly) để tránh OOM
def create_sparse_loader(X_sparse, y_series, batch_size=128, shuffle=True):
    dataset = TensorDataset(torch.arange(X_sparse.shape[0]))
    y_tensor = torch.tensor(y_series.values if hasattr(y_series, 'values') else y_series, dtype=torch.long)

    def collate_fn(batch):
        indices = [item[0].item() for item in batch]
        x_batch = torch.tensor(X_sparse[indices].toarray(), dtype=torch.float32)
        y_batch = y_tensor[indices]
        return x_batch, y_batch

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

BATCH_SIZE = 128
train_loader = create_sparse_loader(X_train_sparse, y_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = create_sparse_loader(X_test_sparse, y_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"Đã khởi tạo train_loader ({len(train_loader)} batches) và test_loader ({len(test_loader)} batches) thành công!")

Đã khởi tạo train_loader (1229 batches) và test_loader (352 batches) thành công!


In [9]:
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_recall_fscore_support, f1_score, average_precision_score

# 1. Hàm huấn luyện Neural Network
def train_nn_model(model, train_loader, epochs=10, lr=1e-3, pos_weight_val=None, device='cpu'):
    model.to(device)
    pos_weight_tensor = torch.tensor([pos_weight_val], device=device) if pos_weight_val else None
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs.squeeze(), batch_y.float())
            loss.backward()
            optimizer.step()

    return model

# 2. Hàm đánh giá mô hình
def eval_nn_model(model, test_loader, model_name, device='cpu'):
    model.eval()
    model.to(device)
    all_preds, all_probs, all_targets = [], [], []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            logits = model(batch_x).squeeze()
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(batch_y.numpy())

    all_targets = np.array(all_targets)
    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)

    p, r, f1, _ = precision_recall_fscore_support(all_targets, all_preds, labels=[0, 1], zero_division=0)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

    # Đã sửa lỗi thiếu dấu trừ
    prob_bad = 1.0 - all_probs
    ap = average_precision_score((all_targets == 0).astype(int), prob_bad)

    return {
        'Method': model_name,
        'Bad-Precision': p[0],
        'Bad-Recall': r[0],
        'Bad-F1': f1[0],
        'Macro-F1': macro_f1,
        'AP': ap
    }

# 3. Huấn luyện Scratch & Transfer Learning
pos_weight_val = float((y_train == 1).sum() / max(1, (y_train == 0).sum()))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Đang chạy huấn luyện trên thiết bị: {device}")

model_scratch_trained = train_nn_model(model_scratch, train_loader, epochs=10, lr=1e-3, pos_weight_val=pos_weight_val, device=device)
model_transfer_trained = train_nn_model(model_transfer, train_loader, epochs=10, lr=1e-3, pos_weight_val=pos_weight_val, device=device)

# 4. Đánh giá và tổng hợp kết quả
results.append(eval_nn_model(model_scratch_trained, test_loader, 'Train from Scratch', device=device))
results.append(eval_nn_model(model_transfer_trained, test_loader, 'Fox8-pretrained + Fine-tuning', device=device))

# 5. Xuất và hiển thị DataFrame
final_e2_results_df = pd.DataFrame(results)

os.makedirs('/content/drive/MyDrive/Colab Notebooks/evaluation', exist_ok=True)
csv_output_path = '/content/drive/MyDrive/Colab Notebooks/evaluation/E2_RESULTS_TABLE.csv'
final_e2_results_df.to_csv(csv_output_path, index=False)

print("=== BẢNG TỔNG HỢP KẾT QUẢ E2 (SCRATCH VS TRANSFER) ===")
display(final_e2_results_df)

Đang chạy huấn luyện trên thiết bị: cuda
=== BẢNG TỔNG HỢP KẾT QUẢ E2 (SCRATCH VS TRANSFER) ===


,Method,Bad-Precision,Bad-Recall,Bad-F1,Macro-F1,AP
0,LR (Full Features),0.089131,0.965147,0.163191,0.559996,0.471164
1,LR (No tweet_count),0.086341,0.981233,0.158716,0.556605,0.368820
2,Train from Scratch,0.996078,0.680965,0.808917,0.903786,0.920416
3,Fox8-pretrained + Fine-tuning,0.969231,0.844504,0.902579,0.950908,0.918244
